# 01 Ingest Bronze

Reads the three raw CSVs and writes them as Delta tables `bronze_subscriptions`, `bronze_payments`, `bronze_chargebacks`.

- Bronze keeps the data as it arrived: every column is read as a string, nothing is cast or dropped. Typing happens in Silver.
- Two lineage columns are added: `_source_file` and `_ingested_at`.
- Tables are overwritten on every run, so a rerun is safe.

Parameters (widgets):
- `CATALOG`: Unity Catalog catalog. Blank means the workspace default catalog.
- `SCHEMA`: target schema. Created if missing.
- `RAW_DIR`: folder that holds `subscriptions/`, `payments/`, `chargebacks/` with CSV files inside. Blank means the Volume `/Volumes/<CATALOG>/<SCHEMA>/raw`.

In [1]:
dbutils.widgets.text("CATALOG", "")
dbutils.widgets.text("SCHEMA", "demo_finance")
dbutils.widgets.text("RAW_DIR", "")

CATALOG = dbutils.widgets.get("CATALOG").strip() or spark.sql("SELECT current_catalog()").first()[0]
SCHEMA = dbutils.widgets.get("SCHEMA").strip()
RAW_DIR = (dbutils.widgets.get("RAW_DIR").strip() or f"/Volumes/{CATALOG}/{SCHEMA}/raw").rstrip("/")
if not SCHEMA:
    raise ValueError("SCHEMA must not be empty")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`")
spark.sql(f"USE `{CATALOG}`.`{SCHEMA}`")
print(f"target schema: {CATALOG}.{SCHEMA}")
print(f"raw dir:       {RAW_DIR}")

target schema: <catalog>.demo_finance_runnable
raw dir:       /Volumes/<catalog>/demo_finance_runnable/raw


In [2]:
from pyspark.sql import functions as F

# Expected columns per source. Ingest fails fast if a file does not match.
SOURCES = {
    "subscriptions": ["customer_id", "plan", "start_date", "end_date", "amount", "status"],
    "payments":      ["payment_id", "customer_id", "ts", "amount", "status", "method"],
    "chargebacks":   ["payment_id", "opened_ts", "reason", "resolved_ts", "outcome"],
}

def read_raw(name: str):
    path = f"{RAW_DIR}/{name}/*.csv"
    df = (spark.read
          .option("header", True)
          .option("inferSchema", False)   # keep raw values as strings
          .csv(path))
    if df.columns != SOURCES[name]:
        raise ValueError(f"{path}: expected columns {SOURCES[name]}, got {df.columns}")
    return (df
            .withColumn("_source_file", F.col("_metadata.file_path"))
            .withColumn("_ingested_at", F.current_timestamp()))

for name in SOURCES:
    target = f"bronze_{name}"
    (read_raw(name).write
        .mode("overwrite")
        .option("overwriteSchema", True)
        .format("delta")
        .saveAsTable(target))
    print(f"{target:<22} {spark.table(target).count():>6} rows")

bronze_subscriptions     1000 rows
bronze_payments         12722 rows
bronze_chargebacks        161 rows


In [3]:
print("Bronze ingest complete.")

Bronze ingest complete.
